# Background to signal $K_SKK$ Flavour tags
## Calculate background to signal ratios for $K_SKK$ vs flavour tags

### Include library for handling uncertainties
#### [Here is the ```uncertainties-cpp``` library on GitHub](https://github.com/Gattocrucco/uncertainties-cpp)

In [1]:
gInterpreter->AddIncludePath("/data/lhcb/users/tat/uncertainties-cpp");

In [2]:
#include<uncertainties/impl.hpp>
#include<uncertainties/ureal.hpp>
#include<uncertainties/io.hpp>
#include<uncertainties/math.hpp>
#include<uncertainties/stat.hpp>

### Load utility functions

In [3]:
gROOT->ProcessLine(".L ../UtilityFunctions.C");

### Number of bins

In [4]:
const int NumberBins = 4;

### Get reconstructed background bin yields

In [5]:
std::map<int, double> GetRecBackgroundBinYields(const std::string &Tag) {
    std::string Filename = "${BES3_ANALYSIS_PATH}/Selection/PeakingBackgrounds/DoubleTag/";
    Filename += Tag + "/KSKK_vs_" + Tag + "_to_KKpipi_vs_" + Tag + "_DoubleTag_SignalMC_Binned.root";
    TChain Chain((Tag + "DoubleTag").c_str());
    Chain.Add(Filename.c_str());
    return GetBinYields(&Chain, true, NumberBins);
}

### Tag modes

In [6]:
std::vector<std::string> Tags{"Kpi", "KeNu"};

### Start calculating the background to signal bin efficiencies, times the ratio of branching fractions

In [7]:
std::string BackgroundToSignalRatios;
for(const auto &Tag : Tags) {
    const auto SignalBF = GetBranchingFraction("KKpipi");
    uncertainties::udouble SignalBF_unc(SignalBF.first, SignalBF.second);
    const auto BackgroundBF = GetBranchingFraction("KSKK");
    uncertainties::udouble BackgroundBF_unc(BackgroundBF.first, BackgroundBF.second);
    const auto BFRatio = BackgroundBF_unc/SignalBF_unc;
    const auto SignalRecYields = GetRecSignalBinYields(Tag, NumberBins);
    const double SignalGenYields = 800000.0;
    const auto BackgroundRecYields = GetRecBackgroundBinYields(Tag);
    const double BackgroundGenYields = 800000.0;
    std::vector<uncertainties::udouble> BkgToSigRatio;
    for(int Bin = -NumberBins; Bin <= NumberBins; Bin++) {
        if(Bin == 0) {
            continue;
        }
        std::string Label = Tag + "_PeakingBackground0_DoubleTag_Flavour_KKpipi_vs_" + Tag + "_SignalBin";
        Label += (Bin > 0 ? "P" : "M") + std::to_string(TMath::Abs(Bin));
        Label += "_TagBin0_BackgroundToSignalRatio";
        const double SigEff = SignalRecYields.at(Bin)/SignalGenYields;
        const double SigEff_err = TMath::Sqrt(SigEff*(1.0 - SigEff)/SignalGenYields);
        const uncertainties::udouble SigEff_unc(SigEff, SigEff_err);
        const double BkgEff = BackgroundRecYields.at(Bin)/BackgroundGenYields;
        const double BkgEff_err = TMath::Sqrt(BkgEff*(1.0 - BkgEff)/BackgroundGenYields);
        const uncertainties::udouble BkgEff_unc(BkgEff, BkgEff_err);
        const auto EffRatio = BkgEff_unc/SigEff_unc;
        BkgToSigRatio.push_back(EffRatio*BFRatio);
        BackgroundToSignalRatios += Label + " ";
        BackgroundToSignalRatios += std::to_string(uncertainties::nom(BkgToSigRatio.back())) + "\n";
        BackgroundToSignalRatios += Label + "_err ";
        BackgroundToSignalRatios += std::to_string(uncertainties::sdev(BkgToSigRatio.back())) + "\n";
    }
    BackgroundToSignalRatios += "\n";
    std::string Filename = "PeakingBackground_DT_KSKK_to_KKpipi_vs_" + Tag + ".root";
    std::vector<double> FlatCovMatrix =
        uncertainties::cov_matrix<std::vector<double>>(BkgToSigRatio);
    SaveCovMatrix(FlatCovMatrix, Filename);
}
std::cout << BackgroundToSignalRatios;

Kpi_PeakingBackground0_DoubleTag_Flavour_KKpipi_vs_Kpi_SignalBinM4_TagBin0_BackgroundToSignalRatio 0.052221
Kpi_PeakingBackground0_DoubleTag_Flavour_KKpipi_vs_Kpi_SignalBinM4_TagBin0_BackgroundToSignalRatio_err 0.006297
Kpi_PeakingBackground0_DoubleTag_Flavour_KKpipi_vs_Kpi_SignalBinM3_TagBin0_BackgroundToSignalRatio 0.006345
Kpi_PeakingBackground0_DoubleTag_Flavour_KKpipi_vs_Kpi_SignalBinM3_TagBin0_BackgroundToSignalRatio_err 0.000970
Kpi_PeakingBackground0_DoubleTag_Flavour_KKpipi_vs_Kpi_SignalBinM2_TagBin0_BackgroundToSignalRatio 0.013866
Kpi_PeakingBackground0_DoubleTag_Flavour_KKpipi_vs_Kpi_SignalBinM2_TagBin0_BackgroundToSignalRatio_err 0.001827
Kpi_PeakingBackground0_DoubleTag_Flavour_KKpipi_vs_Kpi_SignalBinM1_TagBin0_BackgroundToSignalRatio 0.027312
Kpi_PeakingBackground0_DoubleTag_Flavour_KKpipi_vs_Kpi_SignalBinM1_TagBin0_BackgroundToSignalRatio_err 0.003728
Kpi_PeakingBackground0_DoubleTag_Flavour_KKpipi_vs_Kpi_SignalBinP1_TagBin0_BackgroundToSignalRatio 0.011612
Kpi_PeakingB

### Save parameters to a file

In [8]:
std::ofstream File("BackgroundToSignalRatios_Flavour_KSKK.txt");
File << BackgroundToSignalRatios;
File.close();